In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)  # 강제로 .env 다시 로드

import openai, json

from openai.types.chat import ChatCompletionMessage

client = openai.OpenAI()

messages = []

In [2]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_detail(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return response.json()

FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_detail': get_movie_detail,
    'get_similar_movies': get_similar_movies,
    }

In [3]:
TOOLS = [
    {
        "type" : "function",
        "function" : {
            "name" : "get_popular_movies", 
            "description": "Get the list of popular movies in /movies",
            "parameters": {
                "type":"object",
                "properties": {},
                "required": [],
            },
        },
    },
        {
        "type" : "function",
        "function" : {
            "name" : "get_movie_detail", 
            "description": "you can search the credits of the movie by id in /movies/:id/credits",
            "parameters": {
                "type":"object",
                "properties": {
                    "id": {
                        "type":"integer", 
                        "description":"you can search the credits of the movie by id",
                    }
                },
                "required": ["id"],
            },
        },
    },
        {
        "type" : "function",
        "function" : {
            "name" : "get_similar_movies", 
            "description": "you can search the similar movies by id in /movies/:id/similar",
            "parameters": {
                "type":"object",
                "properties": {
                    "id": {
                        "type":"integer", 
                        "description":"you can search the credits of the movie by id",
                    }
                },
                "required": ["id"],
            },
        },
    }
]

In [4]:
def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role":"assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type":"function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Agent:[{function_name}({arguments}) 호출]")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result),
            })
        
        call_ai()
    else:
        messages.append({"role":"assistant", "content":message.content})
        print(f"Agent: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [5]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User:{message}")
        call_ai()

User:지금 인기 있는 영화를 알려줘
Agent:[get_popular_movies({}) 호출]
Ran get_popular_movies
Agent: 현재 인기 있는 영화 목록입니다.

1. **Your Heart Will Be Broken**
   - 개봉일: 2026-03-26
   - 평점: 6.75
   - [포스터 이미지](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)
   - 줄거리: 고등학생 폴리나가 새로운 학교에서 괴롭힘으로부터 구출되고, 주요 괴롭히는 바르스와 데이트를 가장하기로 계약을 맺고 보호받는데, 이를 통해 진정한 감정을 발전시킵니다.

2. **Avatar: Fire and Ash**
   - 개봉일: 2025-12-17
   - 평점: 7.37
   - [포스터 이미지](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)
   - 줄거리: 제이크 술리와 네이티리는 새로운 적인 아쉬 사람들과의 전투에서 생존을 위해 싸워야 합니다.

3. **Hoppers**
   - 개봉일: 2026-03-04
   - 평점: 7.59
   - [포스터 이미지](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)
   - 줄거리: 인간 의식을 생생한 로봇 동물로 '호핑'하는 기술을 통해 애완동물과 소통하는 여정을 그립니다.

4. **Shelter**
   - 개봉일: 2026-01-28
   - 평점: 6.79
   - [포스터 이미지](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)
   - 줄거리: 한 남자가 폭풍에서 소녀를 구하고, 그의 과거와 연결된 적으로부터 그를 보호해야 하는 이야기를 다룹니다.

5. **Crime 101**
   - 개봉일: 2026-02-11
   - 

In [6]:

# PROMPT = """
# I have the following API endpoints in my system.
# All of them receive the following parameters in the API https://nomad-movies.nomadcoders.workers.dev/ :
# /movies - the list of the popular movies
# /movies/:id - you can search the detail of the movie by id
# /movies/:id/credits - you can search the credits of the movie by id
# /movies/:id/similar - you can search the similar movies by id

# You can use the following functions to get the information you need.
# 'get_popular_movies()': Get the list of popular movies in /movies
# 'get_movie_detail(id)': Get the list of credits of the movie
# 'get_similar_movies(movie_id)': Get the list of similar movies

# Please say nothing else, just return the name of the function with the arguments.

# Answer all of the following questions : 

# 1.What is the most popular movie?
# 2.Which movie has the movie id 550?
# 3.Tell me the credits of the movie with the id 550.

# """

# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": PROMPT}],
# )

# response

In [7]:
# message = response.choices[0].message.content
# message